# Accumulated SIRD Model Observations

This notebook builds a daily accumulated series of cases, active infections,
recoveries, deaths, and susceptibles. It uses the midpoint of the recovery
rate for each epidemiological phase to estimate recoveries.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from codes import covid_positive, dates_to_datetime

DATA_DIR = Path.cwd().parent / "Datos"
WAVE_DIR = DATA_DIR / "waves"
OUTPUT_FILE = DATA_DIR / "sird_observations.csv"
POPULATION = 126_014_024

print(DATA_DIR)
print(OUTPUT_FILE)

/Users/marcosbautista/Documents/GitHub/Estimation-of-epidemiological-parameters/Datos
/Users/marcosbautista/Documents/GitHub/Estimation-of-epidemiological-parameters/Datos/sird_observations.csv


In [2]:
phases = [
    {"phase": "Pre-lockdown", "start": "2020-02-28", "end": "2020-03-22",
     "beta": (0.5534, 1.1400), "gamma": (0.0708, 0.2000), "delta": (0.0561, 0.1200),
     "file": "pre_lockdown_filtered_data_2020-02-28_2020-03-22.csv"},
    {"phase": "First Wave", "start": "2020-03-23", "end": "2020-09-30",
     "beta": (0.5534, 0.1069), "gamma": (0.0708, 0.1383), "delta": (0.0561, 0.0211),
     "file": "filtered_data_2020-03-23_2020-09-30.csv"},
    {"phase": "Second Wave", "start": "2020-10-01", "end": "2021-04-30",
     "beta": (0.3500, 0.5000), "gamma": (0.1200, 0.1500), "delta": (0.0150, 0.0250),
     "file": "filtered_data_2020-10-01_2021-04-30.csv"},
    {"phase": "Third Wave", "start": "2021-05-01", "end": "2021-11-30",
     "beta": (0.6000, 0.8500), "gamma": (0.1400, 0.1800), "delta": (0.0080, 0.0150),
     "file": "filtered_data_2021-05-01_2021-11-30.csv"},
    {"phase": "Fourth Wave", "start": "2021-12-01", "end": "2022-03-31",
     "beta": (1.1000, 1.6000), "gamma": (0.2000, 0.3300), "delta": (0.0010, 0.0030),
     "file": "filtered_data_2021-12-01_2022-03-31.csv"},
    {"phase": "Fifth Wave", "start": "2022-05-01", "end": "2022-10-31",
     "beta": (0.9000, 1.3000), "gamma": (0.2200, 0.3000), "delta": (0.0005, 0.0015),
     "file": "filtered_data_2022-05-01_2022-10-31.csv"},
    {"phase": "Sixth Wave", "start": "2022-11-01", "end": "2023-03-31",
     "beta": (0.8000, 1.2000), "gamma": (0.2500, 0.3300), "delta": (0.0003, 0.0010),
     "file": "filtered_data_2022-11-01_2023-03-31.csv"},
]

def midpoint(values):
    return sum(values) / len(values)

for wave in phases:
    wave["beta_mid"] = midpoint(wave["beta"])
    wave["gamma_mid"] = midpoint(wave["gamma"])
    wave["delta_mid"] = midpoint(wave["delta"])
    wave["recovery_days"] = round(1.0 / wave["gamma_mid"])

parameter_table = pd.DataFrame([{
    "Epidemiological Phase": wave["phase"],
    "Start Date": wave["start"],
    "End Date": wave["end"],
    "Midpoint beta": wave["beta_mid"],
    "Midpoint gamma": wave["gamma_mid"],
    "Midpoint delta": wave["delta_mid"],
    "Recovery Days": wave["recovery_days"],
} for wave in phases])

parameter_table

,Epidemiological Phase,Start Date,End Date,Midpoint beta,Midpoint gamma,Midpoint delta,Recovery Days
0,Pre-lockdown,2020-02-28,2020-03-22,0.84670,0.13540,0.08805,7
1,First Wave,2020-03-23,2020-09-30,0.33015,0.10455,0.03860,10
2,Second Wave,2020-10-01,2021-04-30,0.42500,0.13500,0.02000,7
3,Third Wave,2021-05-01,2021-11-30,0.72500,0.16000,0.01150,6
4,Fourth Wave,2021-12-01,2022-03-31,1.35000,0.26500,0.00200,4
5,Fifth Wave,2022-05-01,2022-10-31,1.10000,0.26000,0.00100,4
6,Sixth Wave,2022-11-01,2023-03-31,1.00000,0.29000,0.00065,3


In [3]:
blocks = []
for wave in phases:
    file = WAVE_DIR / wave["file"]
    if not file.exists():
        raise FileNotFoundError(file)

    block = pd.read_csv(
        file,
        usecols=["FECHA_INGRESO", "FECHA_DEF", "CLASIFICACION_FINAL"],
        low_memory=False,
    )
    block["phase"] = wave["phase"]
    blocks.append(block)

records = pd.concat(blocks, ignore_index=True)
records = records.drop_duplicates()
records = dates_to_datetime(records, ["FECHA_INGRESO"])
positive = covid_positive(records)

print(f"Loaded records: {len(records):,}")
print(f"Positive cases: {len(positive):,}")
print(f"Coverage: {positive['FECHA_INGRESO'].min().date()} to {positive['FECHA_INGRESO'].max().date()}")

Loaded records: 78,924
Positive cases: 42,621
Coverage: 2020-02-28 to 2023-03-31


In [4]:
date_1 = pd.Timestamp("2020-02-28")
date_f = pd.Timestamp("2023-03-31")
index = pd.date_range(date_1, date_f, freq="D")

# Accumulated cases by admission date.
new_cases = positive.groupby("FECHA_INGRESO").size().reindex(index, fill_value=0)
C = new_cases.cumsum()

# Accumulated deaths by registered death date.
death_dates = pd.to_datetime(positive["FECHA_DEF"], errors="coerce")
new_deaths = death_dates.value_counts().reindex(index, fill_value=0)
D = new_deaths.cumsum()

# Recovery: assign each survivor a recovery date calculated with
# the midpoint gamma corresponding to the admission date.
survivors = positive[death_dates.isna()].copy()
gamma_series = pd.Series(np.nan, index=index, dtype=float)
gamma_phase = pd.Series(pd.NA, index=index, dtype="object")

for wave in phases:
    start = pd.Timestamp(wave["start"])
    end = pd.Timestamp(wave["end"])
    mask = (index >= start) & (index <= end)
    gamma_series.loc[mask] = wave["gamma_mid"]
    gamma_phase.loc[mask] = wave["phase"]

# The official intervals do not include April 2022. Preserve the value
# from the previous phase until the fifth wave starts.
gamma_series = gamma_series.ffill().bfill()
gamma_phase = gamma_phase.ffill().bfill()

survivor_dates = survivors["FECHA_INGRESO"].copy()
survivor_dates = survivor_dates[survivor_dates.between(date_1, date_f, inclusive="both")]
recovery_days = survivor_dates.map(gamma_series.to_dict()).map(lambda gamma: round(1.0 / gamma))
recovery_dates = survivor_dates + pd.to_timedelta(recovery_days, unit="D")

R = recovery_dates.value_counts().reindex(index, fill_value=0).cumsum()
I = C - R - D
S = POPULATION - C

observations = pd.DataFrame({
    "Date": index,
    "S": S.astype(float),
    "I": I.astype(float),
    "R": R.astype(float),
    "D": D.astype(float),
    "C": C.astype(float),
    "Epidemiological Phase": gamma_phase,
    "Midpoint gamma": index.map(gamma_series.to_dict()),
    "Recovery Days": index.map(gamma_series.to_dict()).map(lambda gamma: round(1.0 / gamma)),
}, index=index)

# Starting in April 2022, metadata corresponds to the active phase.
beta_by_date = pd.Series(np.nan, index=index, dtype=float)
delta_by_date = pd.Series(np.nan, index=index, dtype=float)
for wave in phases:
    start = pd.Timestamp(wave["start"])
    end = pd.Timestamp(wave["end"])
    mask = (index >= start) & (index <= end)
    beta_by_date.loc[mask] = wave["beta_mid"]
    delta_by_date.loc[mask] = wave["delta_mid"]
beta_by_date = beta_by_date.ffill().bfill()
delta_by_date = delta_by_date.ffill().bfill()
observations["Midpoint beta"] = beta_by_date
observations["Midpoint delta"] = delta_by_date

observations.to_csv(OUTPUT_FILE, index=False)

print(f"Generated file: {OUTPUT_FILE}")
observations.tail(10)

Generated file: /Users/marcosbautista/Documents/GitHub/Estimation-of-epidemiological-parameters/Datos/sird_observations.csv


,Date,S,I,R,D,C,Epidemiological Phase,Midpoint gamma,Recovery Days,Midpoint beta,Midpoint delta
2023-03-22,2023-03-22,125971505.0,104.0,2203.0,40212.0,42519.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-23,2023-03-23,125971496.0,99.0,2205.0,40224.0,42528.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-24,2023-03-24,125971482.0,100.0,2207.0,40235.0,42542.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-25,2023-03-25,125971470.0,101.0,2209.0,40244.0,42554.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-26,2023-03-26,125971462.0,95.0,2211.0,40256.0,42562.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-27,2023-03-27,125971446.0,98.0,2213.0,40267.0,42578.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-28,2023-03-28,125971436.0,98.0,2215.0,40275.0,42588.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-29,2023-03-29,125971425.0,100.0,2217.0,40282.0,42599.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-30,2023-03-30,125971415.0,102.0,2219.0,40288.0,42609.0,Sixth Wave,0.29,3,1.0,0.00065
2023-03-31,2023-03-31,125971403.0,105.0,2221.0,40295.0,42621.0,Sixth Wave,0.29,3,1.0,0.00065


In [5]:
observation_error = (
    observations["S"] + observations["I"] +
    observations["R"] + observations["D"] - POPULATION
).abs().max()

print(f"Maximum conservation error: {observation_error:.6f}")
print(f"Minimum I: {observations['I'].min():,.2f}")
print(f"Minimum R: {observations['R'].min():,.2f}")
print(f"Minimum D: {observations['D'].min():,.2f}")
print(f"Rows: {len(observations):,}")

assert np.isclose(observation_error, 0.0)
assert (observations[["S", "I", "R", "D", "C"]] >= 0).all().all()
assert len(observations) == (date_f - date_1).days + 1

Maximum conservation error: 0.000000
Minimum I: 1.00
Minimum R: 0.00
Minimum D: 0.00
Rows: 1,128
